# Regresión lineal múltiple con NumPy

En este notebook vamos a adaptar el ejemplo de regresión lineal para una base de datos con **3 variables de entrada**:

- `x1`
- `x2`
- `x3`

Y una variable objetivo:

$$y = 1 + 2x_1 + x_2 + 2x_3$$

La idea es entrenar un modelo para que aprenda una función parecida a esta:

$$h(x) = w_0 + w_1x_1 + w_2x_2 + w_3x_3$$

Donde:

- `w0` es el intercepto o sesgo.
- `w1`, `w2` y `w3` son los pesos de cada variable.

## 1. Importar librerías

Usaremos principalmente `NumPy`, que permite trabajar con arreglos numéricos y hacer operaciones matemáticas de forma eficiente.

También usaremos `pandas` solo para mostrar la tabla de datos de manera más ordenada.

In [ ]:
import numpy as np
import pandas as pd

## 2. Crear la base de datos

La tabla tiene 7 ejemplos. Cada fila contiene los valores de `x1`, `x2`, `x3` y el resultado esperado `y`.

La fórmula usada para calcular `y` es:

$$y = 1 + 2x_1 + x_2 + 2x_3$$

In [ ]:
# Datos de entrada
x1 = np.array([1, 2, 3, 4, 5, 6, 7], dtype=np.float32)
x2 = np.array([2, 3, 4, 5, 6, 7, 8], dtype=np.float32)
x3 = np.array([6, 7, 6, 7, 6, 7, 6], dtype=np.float32)

# Variable objetivo
Y = np.array([17, 22, 23, 28, 29, 34, 35], dtype=np.float32)

# Mostrar los datos en una tabla
tabla = pd.DataFrame({
    "x1": x1,
    "x2": x2,
    "x3": x3,
    "y": Y
})

tabla

## 3. Unir las variables de entrada

Para entrenar el modelo, juntamos `x1`, `x2` y `x3` en una sola matriz llamada `X`.

Cada fila representa un ejemplo, y cada columna representa una característica o variable independiente.

In [ ]:
X = np.column_stack((x1, x2, x3))

print("Matriz X:")
print(X)

print("\nVector Y:")
print(Y)

## 4. Definir el modelo

El modelo de regresión lineal múltiple será:

$$h(x) = w_0 + w_1x_1 + w_2x_2 + w_3x_3$$

Al inicio, los pesos estarán en cero. Durante el entrenamiento, el algoritmo irá ajustando esos valores.

In [ ]:
# Pesos iniciales
w0 = 0.0
w1 = 0.0
w2 = 0.0
w3 = 0.0

# Funcion de prediccion
def forward(X):
    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = X[:, 2]
    return w0 + w1*x1 + w2*x2 + w3*x3

print("Predicciones iniciales:")
print(forward(X))

## 5. Definir la función de pérdida

La función de pérdida mide qué tan lejos están las predicciones del modelo respecto a los valores reales.

Usaremos el error cuadrático medio dividido entre 2:

$$J = \frac{1}{2m}\sum (\hat{y} - y)^2$$

Si la pérdida es grande, el modelo está prediciendo mal. Si la pérdida se acerca a cero, el modelo está aprendiendo bien.

In [ ]:
def loss(y, y_pred):
    return ((y_pred - y) ** 2).mean() / 2

pred_inicial = forward(X)
print("Perdida inicial:", loss(Y, pred_inicial))

## 6. Calcular gradientes

Los gradientes indican cuánto debe cambiar cada peso para reducir el error.

Para este modelo tenemos cuatro gradientes:

- Gradiente de `w0`
- Gradiente de `w1`
- Gradiente de `w2`
- Gradiente de `w3`

Luego usaremos esos gradientes en el algoritmo de **descenso por gradiente**.

In [ ]:
def gradient(X, y, y_pred):
    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = X[:, 2]

    error = y_pred - y

    dw0 = error.mean()
    dw1 = (error * x1).mean()
    dw2 = (error * x2).mean()
    dw3 = (error * x3).mean()

    return dw0, dw1, dw2, dw3

## 7. Entrenar el modelo

Ahora entrenamos usando descenso por gradiente.

En cada época el modelo hace lo siguiente:

1. Calcula las predicciones.
2. Calcula la pérdida.
3. Calcula los gradientes.
4. Actualiza los pesos.

La tasa de aprendizaje (`learning_rate`) controla qué tan grandes son los cambios en los pesos.

In [ ]:
learning_rate = 0.001
n_iters = 5000

historial_loss = []

for epoch in range(n_iters):
    # 1. Prediccion
    y_pred = forward(X)

    # 2. Perdida
    l = loss(Y, y_pred)
    historial_loss.append(l)

    # 3. Gradientes
    dw0, dw1, dw2, dw3 = gradient(X, Y, y_pred)

    # 4. Actualizacion de pesos
    w0 -= learning_rate * dw0
    w1 -= learning_rate * dw1
    w2 -= learning_rate * dw2
    w3 -= learning_rate * dw3

    if epoch % 500 == 0:
        print(
            f"epoch {epoch+1}: "
            f"w0={w0:.4f}, w1={w1:.4f}, w2={w2:.4f}, w3={w3:.4f}, "
            f"loss={l:.6f}"
        )

## 8. Ver los pesos aprendidos

La fórmula real era:

$$y = 1 + 2x_1 + x_2 + 2x_3$$

Por lo tanto, idealmente esperamos algo parecido a:

- `w0 ≈ 1`
- `w1 ≈ 2`
- `w2 ≈ 1`
- `w3 ≈ 2`

Sin embargo, como `x2` depende directamente de `x1` porque `x2 = x1 + 1`, puede haber más de una combinación de pesos que produzca predicciones correctas. Eso se llama **colinealidad**.

In [ ]:
print("Pesos finales aprendidos:")
print(f"w0 = {w0:.4f}")
print(f"w1 = {w1:.4f}")
print(f"w2 = {w2:.4f}")
print(f"w3 = {w3:.4f}")

## 9. Comparar valores reales y predichos

Ahora revisamos si el modelo predice bien los valores de la tabla original.

In [ ]:
y_pred_final = forward(X)

resultados = pd.DataFrame({
    "x1": X[:, 0],
    "x2": X[:, 1],
    "x3": X[:, 2],
    "Y real": Y,
    "Y predicho": y_pred_final,
    "Error": Y - y_pred_final
})

resultados

## 10. Graficar la pérdida

Esta gráfica muestra cómo va disminuyendo el error durante el entrenamiento.

Si la curva baja, significa que el modelo está aprendiendo.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(historial_loss)
plt.title("Disminucion de la perdida durante el entrenamiento")
plt.xlabel("Epoca")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

## 11. Hacer una nueva predicción

Probemos el modelo con un nuevo dato:

- `x1 = 8`
- `x2 = 9`
- `x3 = 7`

Con la fórmula original:

$$y = 1 + 2(8) + 9 + 2(7) = 40$$

El modelo debería predecir un valor cercano a 40.

In [ ]:
nuevo_dato = np.array([[8, 9, 7]], dtype=np.float32)
prediccion = forward(nuevo_dato)

print("Prediccion para x1=8, x2=9, x3=7:")
print(prediccion[0])

## 12. Conclusión

En este notebook construimos una regresión lineal múltiple desde cero usando NumPy.

El modelo aprendió a relacionar tres variables de entrada (`x1`, `x2`, `x3`) con una salida `y`.

La función que buscábamos aproximar era:

$$y = 1 + 2x_1 + x_2 + 2x_3$$

Aunque los pesos finales pueden no ser exactamente iguales a `1`, `2`, `1` y `2`, las predicciones pueden ser muy cercanas debido a que algunas variables están relacionadas entre sí.